 Install & Import Libraries

In [1]:
!pip install nltk -q

In [2]:
import re
import nltk
from collections import Counter

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print('All libraries loaded successfully.')

All libraries loaded successfully.


Task 1: Conceptual Understanding

### Q1. What is the difference between "Love" and "love" in NLP?

In NLP, **"Love"** and **"love"** are treated as two completely different tokens by default because computers compare characters exactly as they are. The uppercase "L" and lowercase "l" have different ASCII values, so the model sees them as different words.

However, semantically they mean the same thing. This is why **lowercasing** is one of the first steps in any NLP preprocessing pipeline — it ensures that "Love", "LOVE", and "love" are all treated as the same word, reducing vocabulary size and improving model performance.

---

### Q2. What happens if stopwords are not removed?

If stopwords (common words like "is", "the", "a", "and", "of") are not removed:

- They dominate the **word frequency counts** and overshadow meaningful words
- The model gives **too much weight** to words that carry no real meaning
- The **vocabulary size increases** unnecessarily, which increases computation time
- **Feature vectors** become noisy and less informative for classification or clustering tasks
- The overall **model accuracy decreases** because the signal is buried under noise

---

### Q3. Two real-world scenarios where removing stopwords can be harmful

**Scenario 1 — Sentiment Analysis:**  
The sentence *"I am NOT happy"* versus *"I am happy"* — if we remove the stopword "not", both sentences become *"happy"* and the model completely loses the negative sentiment. This leads to wrong predictions.

**Scenario 2 — Question Answering / Search:**  
A query like *"Who is the president of India?"* — if "is" and "the" are removed, it becomes *"president India"* which loses the grammatical structure and context needed to understand it is a question about a current role.

---

### Q4. Difference between Stemming and Lemmatization

| Feature | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off word endings using rules | Uses a dictionary to find the base form |
| Output | May not be a real word | Always returns a valid word |
| Example | "running" → "run" | "running" → "run" |
| Example | "studies" → "studi" | "studies" → "study" |
| Speed | Faster | Slower |
| Accuracy | Lower | Higher |

**Stemming** is a crude, rule-based approach that just cuts off suffixes — it is fast but can produce non-words like "studi" or "happi".  
**Lemmatization** is smarter — it looks up the actual root form (lemma) of a word using a vocabulary and grammar rules, always producing a valid word.

Task 2: Build Advanced Preprocessing Function

In [3]:
STOP_WORDS = set(stopwords.words('english'))

# Keep these meaningful short words even though length <= 2
KEEP_SHORT = {'no', 'not'}

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """
    Advanced NLP preprocessing pipeline.
    Steps:
      1. Remove URLs and email-like patterns
      2. Remove emojis and non-ASCII characters
      3. Convert to lowercase
      4. Handle repeated characters (e.g. 'sooo' -> 'so')
      5. Remove numbers
      6. Remove punctuation and special characters
      7. Remove extra spaces
      8. Tokenize
      9. Remove stopwords (keeping 'no', 'not')
     10. Remove very short tokens (length <= 2), keeping 'no', 'not'
     11. Lemmatize tokens
    Returns: (tokens list, cleaned sentence string)
    """

    # Handle edge cases
    if not text or not isinstance(text, str):
        return [], ""

    # Step 1: Remove URLs (http, https, www)
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+\.\S+', '', text)

    # Step 2: Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Step 3: Convert to lowercase
    text = text.lower()

    # Step 4: Handle repeated characters — reduce 3+ repeats to 1
    # e.g. 'soooo' -> 'so', 'nooooo' -> 'no', 'baaad' -> 'bad'
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Step 5: Remove numbers
    text = re.sub(r'\d+', '', text)

    # Step 6: Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Step 7: Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 8: Tokenize
    tokens = text.split()

    # Step 9 & 10: Remove stopwords and short tokens, keep 'no', 'not'
    tokens = [
        t for t in tokens
        if t in KEEP_SHORT
        or (t not in STOP_WORDS and len(t) > 2)
    ]

    # Step 11: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    cleaned_sentence = ' '.join(tokens)

    return tokens, cleaned_sentence


print('preprocess_text() function defined successfully.')

preprocess_text() function defined successfully.


Task 3: Stress Testing

In [4]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print('=' * 70)
print('STRESS TEST RESULTS')
print('=' * 70)

results = []

for i, sentence in enumerate(test_sentences, 1):
    tokens, cleaned = preprocess_text(sentence)
    results.append({
        'original':  sentence,
        'tokens':    tokens,
        'cleaned':   cleaned
    })
    print(f'\nSentence {i}:')
    print(f'  Original : {sentence}')
    print(f'  Tokens   : {tokens}')
    print(f'  Cleaned  : {cleaned}')
    print('-' * 70)

STRESS TEST RESULTS

Sentence 1:
  Original : Get 100% FREE access now!!!
  Tokens   : ['get', 'free', 'access']
  Cleaned  : get free access
----------------------------------------------------------------------

Sentence 2:
  Original : I absolutely looooved this product 😍😍
  Tokens   : ['absolutely', 'loved', 'product']
  Cleaned  : absolutely loved product
----------------------------------------------------------------------

Sentence 3:
  Original : Worst service ever... 0/10
  Tokens   : ['worst', 'service', 'ever']
  Cleaned  : worst service ever
----------------------------------------------------------------------

Sentence 4:
  Original : Call me at 9876543210
  Tokens   : ['call']
  Cleaned  : call
----------------------------------------------------------------------

Sentence 5:
  Original : This is THE best course!!!
  Tokens   : ['best', 'course']
  Cleaned  : best course
----------------------------------------------------------------------

Sentence 6:
  Original : Vi

Task 4: Token Analytics

In [7]:
print('=' * 70)
print('TOKEN ANALYTICS')
print('=' * 70)

noise_scores  = []
retain_scores = []

for i, r in enumerate(results, 1):
    tokens       = r['tokens']
    original_len = len(r['original'].split())
    total        = len(tokens)
    unique       = len(set(tokens))
    avg_len      = round(sum(len(t) for t in tokens) / total, 2) if total > 0 else 0
    noise        = original_len - total

    noise_scores.append((i, r['original'], noise))
    retain_scores.append((i, r['original'], total))

    print(f'\nSentence {i}: "{r["original"][:45]}..."' if len(r['original']) > 45 else f'\nSentence {i}: "{r["original"]}"')
    print(f'  Total tokens   : {total}')
    print(f'  Unique tokens  : {unique}')
    print(f'  Avg token len  : {avg_len}')

print('\n' + '=' * 70)

most_noisy   = max(noise_scores,  key=lambda x: x[2])
most_retained = max(retain_scores, key=lambda x: x[2])

print(f'\n Most Noisy Sentence     : Sentence {most_noisy[0]}')
print(f'   "{most_noisy[1]}"')
print(f'   Noise removed: {most_noisy[2]} words')

print(f'\n Most Meaningful Retained: Sentence {most_retained[0]}')
print(f'   "{most_retained[1]}"')
print(f'   Tokens retained: {most_retained[2]} tokens')

TOKEN ANALYTICS

Sentence 1: "Get 100% FREE access now!!!"
  Total tokens   : 3
  Unique tokens  : 3
  Avg token len  : 4.33

Sentence 2: "I absolutely looooved this product 😍😍"
  Total tokens   : 3
  Unique tokens  : 3
  Avg token len  : 7.33

Sentence 3: "Worst service ever... 0/10"
  Total tokens   : 3
  Unique tokens  : 3
  Avg token len  : 5.33

Sentence 4: "Call me at 9876543210"
  Total tokens   : 1
  Unique tokens  : 1
  Avg token len  : 4.0

Sentence 5: "This is THE best course!!!"
  Total tokens   : 2
  Unique tokens  : 2
  Avg token len  : 5.0

Sentence 6: "Visit https://openai.com now!"
  Total tokens   : 1
  Unique tokens  : 1
  Avg token len  : 5.0

Sentence 7: "Nooooo this is baaad!!!"
  Total tokens   : 2
  Unique tokens  : 2
  Avg token len  : 2.5

Sentence 8: "OK OK OK I got it"
  Total tokens   : 1
  Unique tokens  : 1
  Avg token len  : 3.0

Sentence 9: "Win $$$ now!!! Limited offer!!!"
  Total tokens   : 3
  Unique tokens  : 3
  Avg token len  : 5.0

Sentence 10: "

Task 5: Frequency Analysis

In [9]:
all_tokens = []
for r in results:
    all_tokens.extend(r['tokens'])

word_freq = Counter(all_tokens)


print('FREQUENCY ANALYSIS')


print('\n Top 10 Most Frequent Words:')
print(f'  {"Word":<20} {"Count"}')
for word, count in word_freq.most_common(10):
    print(f'  {word:<20} {count}')

print('\n Top 5 Least Frequent Words:')
print(f'  {"Word":<20} {"Count"}')
least_common = word_freq.most_common()[:-6:-1]
for word, count in least_common:
    print(f'  {word:<20} {count}')

print(f'\n  Total unique words across all sentences: {len(word_freq)}')
print(f'  Total tokens across all sentences      : {sum(word_freq.values())}')

FREQUENCY ANALYSIS

 Top 10 Most Frequent Words:
  Word                 Count
  get                  1
  free                 1
  access               1
  absolutely           1
  loved                1
  product              1
  worst                1
  service              1
  ever                 1
  call                 1

 Top 5 Least Frequent Words:
  Word                 Count
  happy                1
  not                  1
  offer                1
  limited              1
  win                  1

  Total unique words across all sentences: 21
  Total tokens across all sentences      : 21


Task 6: Build Full Pipeline

In [10]:
def full_pipeline(text_list):
    """
    Full NLP preprocessing pipeline.
    Takes a list of raw text strings.
    Returns a dict with:
      - 'tokens'          : list of token lists per sentence
      - 'clean_sentences' : list of cleaned sentence strings
      - 'all_tokens'      : flat list of all tokens combined
      - 'word_frequency'  : Counter of all tokens
    """

    if not text_list or not isinstance(text_list, list):
        return {'tokens': [], 'clean_sentences': []}

    all_tokens      = []
    tokens_list     = []
    clean_sentences = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        tokens_list.append(tokens)
        clean_sentences.append(cleaned)
        all_tokens.extend(tokens)

    return {
        'tokens':          tokens_list,
        'clean_sentences': clean_sentences,
        'all_tokens':      all_tokens,
        'word_frequency':  Counter(all_tokens)
    }


# Run pipeline on test sentences
pipeline_output = full_pipeline(test_sentences)

print('FULL PIPELINE OUTPUT')

print('\nTokens per sentence:')
for i, t in enumerate(pipeline_output['tokens'], 1):
    print(f'  Sentence {i}: {t}')

print('\nCleaned sentences:')
for i, s in enumerate(pipeline_output['clean_sentences'], 1):
    print(f'  Sentence {i}: "{s}"')

print(f'\nTotal tokens in pipeline : {len(pipeline_output["all_tokens"])}')
print(f'Unique tokens in pipeline: {len(pipeline_output["word_frequency"])}')

FULL PIPELINE OUTPUT

Tokens per sentence:
  Sentence 1: ['get', 'free', 'access']
  Sentence 2: ['absolutely', 'loved', 'product']
  Sentence 3: ['worst', 'service', 'ever']
  Sentence 4: ['call']
  Sentence 5: ['best', 'course']
  Sentence 6: ['visit']
  Sentence 7: ['no', 'bad']
  Sentence 8: ['got']
  Sentence 9: ['win', 'limited', 'offer']
  Sentence 10: ['not', 'happy']

Cleaned sentences:
  Sentence 1: "get free access"
  Sentence 2: "absolutely loved product"
  Sentence 3: "worst service ever"
  Sentence 4: "call"
  Sentence 5: "best course"
  Sentence 6: "visit"
  Sentence 7: "no bad"
  Sentence 8: "got"
  Sentence 9: "win limited offer"
  Sentence 10: "not happy"

Total tokens in pipeline : 21
Unique tokens in pipeline: 21


Task 7: Error Handling

In [11]:
edge_cases = [
    "",              # Empty string
    "😍😍😍😍😍",    # Only emojis
    "1234567890",    # Only numbers
    "   ",           # Only spaces
    "!!! @@@ ###",   # Only special characters
    None,            # None input
]

print('=' * 70)
print('ERROR HANDLING — EDGE CASES')
print('=' * 70)

for case in edge_cases:
    tokens, cleaned = preprocess_text(case)
    print(f'\nInput   : {repr(case)}')
    print(f'Tokens  : {tokens}')
    print(f'Cleaned : "{cleaned}"')
    print(f'Status  : {"✅ Handled gracefully" if tokens == [] else "✅ Processed"}')
    print('-' * 40)

ERROR HANDLING — EDGE CASES

Input   : ''
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------

Input   : '😍😍😍😍😍'
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------

Input   : '1234567890'
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------

Input   : '   '
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------

Input   : '!!! @@@ ###'
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------

Input   : None
Tokens  : []
Cleaned : ""
Status  : ✅ Handled gracefully
----------------------------------------
